# Temporal alignment within each species (DeST-OT)

Which coupling best links each section to the next stage's section of the same species? This notebook evaluates the output of `bench/scripts/temporal_align.py` (solver and metrics in `mgw/bench/temporal.py`). It reads only saved files and needs no GPU.

**Pairs:** adjacent sections of `bench/configs/chains.yaml` (H1–H6 human, M1–M6 mouse).

**DeST-OT** (Halmos et al. 2025): semi-relaxed OT, every t2 tile has an ancestor, a t1 tile may grow or shrink (KL weight γ). Objective (1−α)·⟨C,P⟩ + α·[β·triplet + (1−β)·quartet] on merged feature-spatial matrices M = C_intra ⊙ D_intra, entropy ε. The growth of tile i is ρ_i = (P1)_i / g1_i − 1. Tile masses are their bins, so the t2 total is the area ratio and the global growth is observed, not imposed.

**Baselines:** expression-only (α = 0, the paper's Expr-only), organ-label semi-relaxed W-OT, balanced FGW (PASTE-like, uniform growth), moscot SpatioTemporalProblem (unbalanced FGW, marker-gene marginals).

**Metrics** (lower is better for the first two):
- **growth distortion** `growth_dist_nt`: how far each tile's growth is from its organ's observed growth m2(p)/m1(p) − 1 (no transitions; the paper's kidney setting). The variant with inferred transitions (`growth_dist`) is the within-organ variance of growth, which a coupling without growth (γ → ∞) makes 0, so it is reported but not used to select.
- **migration**: mean distance from a t1 tile to its descendants after a P-weighted rigid fit (reflections allowed), in t1 RMS radii.
- **κ_bal t2 → t1**: does each t2 tile descend from its own organ? **gene r**: held-out genes transported through P, minus a shuffle.

**The rule** (fixed before looking at the results):
1. Only finite runs that settled count (DeST-OT: P changes by less than `GAP_OK` per refresh at the end; the strict `converged` flag is shown in §2).
2. A DeST-OT setting is *on the front* of a pair if no other run is better by more than `DELTA` (min–max normalised within the pair) in both growth distortion and migration. Settings on the front in at least `FRONT_MIN` of the pairs are eligible.
3. Among them, the setting with the best mean normalised κ_bal and gene r wins: one global setting for all pairs.
4. The label baseline uses the organ labels the metrics score, so it is shown for reference but never compared on κ or growth distortion.

In [ ]:
import sys
sys.path.append("..")

In [ ]:
ROOT = None        # bench root; None: $MGW_BENCH_DIR or <data dir>/bench
RES = 5000         # reduce.max_obs of the run
RUN_DIR = None     # the results folder <norm>_<masses>-<hash>; None: the one with the latest sweep under n<RES>
SEED = 0           # the sweep's seed
DELTA = 0.05       # front tolerance, in min-max units within a pair
FRONT_MIN = 0.75   # share of pairs a setting must be on the front of
GAP_OK = 5e-4      # DeST-OT / expr / label runs are usable when P changed by less than this per refresh at the end

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.lines import Line2D

from mgw import stack3d as s3
from mgw.bench import metrics, prepare, temporal as tp
from mgw.bench.report.figures import OTHER, SLOTS, organ_palette

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 300)
base = prepare.bench_root(ROOT) / "diag" / "temporal" / f"n{RES}"
# the folder whose sweep table was written last (or RUN_DIR)
OUT = base / RUN_DIR if RUN_DIR else max((d for d in base.iterdir() if tp.runs_dir(d, SEED).is_dir()),
                                         key=lambda d: max(f.stat().st_mtime for f in tp.runs_dir(d, SEED).glob("*.csv")))
pairs = pd.read_csv(OUT / "pairs.csv")
runs = tp.load_runs(OUT, SEED)
view = np.load(OUT / f"view_seed{SEED}.npz", allow_pickle=False)
ORGANS = view["organs"].astype(str)
final = tp.load_final(OUT)
settings = json.loads((OUT / "settings.json").read_text())
PAIRS = [p for p in pairs[pairs.seed == SEED].pair if p in set(runs.pair)]     # pairs with sweep results
pinfo = pairs[pairs.seed == SEED].set_index("pair")

# light surface and ink as in label_alignment_best; sequential blue; categorical slots per method
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
SEQ = LinearSegmentedColormap.from_list("blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
DIV = LinearSegmentedColormap.from_list("div", ["#184f95", "#6da7ec", "#f2f1ee", "#f0a070", "#c4431a"])
METHOD = {"destot": (SLOTS[0], "o", "DeST-OT"), "expr": (SLOTS[1], "s", "expression only"),
          "label": (SLOTS[3], "D", "labels (reference)"), "fgw_bal": (SLOTS[2], "^", "balanced FGW"),
          "moscot": (SLOTS[6], "v", "moscot")}
GAMMA_MARK = dict(zip(sorted(runs.gamma.dropna().unique()), "o^sDvP*X"))


def style(ax):
    ax.grid(color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, length=0, labelsize=8)
    for s in ax.spines.values():
        s.set_visible(False)


def grid_axes(n, ncol=4, w=3.4, h=3.0):
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(w * ncol, h * nrow), squeeze=False)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, list(axes.flat)


for c in ("outer_stopped", "gap"):
    if c not in runs:
        runs[c] = np.nan
# usable: DeST-OT, expr, label: P settled (gap <= GAP_OK); FGW: the outer loop stopped before its cap (OTT's flag
# also needs every inner Sinkhorn at 1e-7, which float32 rarely reaches); moscot: its flag
settled = np.where(runs.method.isin(["destot", "expr", "label"]), runs.gap <= GAP_OK,
                   np.where(runs.method == "fgw_bal", runs.outer_stopped.fillna(False).astype(bool),
                            runs.converged.fillna(False).astype(bool)))
ok = runs.error.isna() & runs.finite.fillna(False).astype(bool) & settled
runs["ok"] = ok
print(f"{OUT}\n{len(PAIRS)} pairs, {runs.run.nunique()} runs, {len(runs)} rows ({(~ok).sum()} not usable)")
print(json.dumps(settings, indent=None)[:400])

## 1. Pairs

Stages, days between them, tiles and the observed area ratio Σg2 (t2 bins / t1 bins). The growth units assume both sections have the same bin pitch, so a pitch ratio far from 1 would make the area ratio meaningless.

In [ ]:
t = pinfo[["species", "a", "b", "dt", "n_a", "n_b", "bins_a", "bins_b", "mass_t2", "pitch_a", "pitch_b", "n_shared_genes"]].copy()
t["pitch_ratio"] = t.pitch_b / t.pitch_a
print(t.round(3).to_string())
bad = t[(t.pitch_ratio - 1).abs() > 0.1]
print("\nbin pitch differs by > 10%: " + (", ".join(bad.index) if len(bad) else "none"))

## 2. Solver health

Converged share, failures and run time per method. DeST-OT's convergence: the relative change of P between gradient refreshes falls below `solver.tol` at the final ε (proximal steps and ε-annealing, see `mgw/bench/temporal.py`). FGW reports OTT's flag, which also requires every inner Sinkhorn to converge.

In [ ]:
h = runs.groupby("method").agg(rows=("run", "size"), converged=("converged", "mean"), usable=("ok", "mean"),
                               failed=("error", lambda e: e.notna().sum()),
                               col_err_max=("col_err", "max"), seconds_median=("seconds_total", "median"),
                               seconds_total=("seconds_total", "sum"))
print(h.round(4).to_string())
d = runs[runs.method == "destot"]
if len(d):
    print("\nDeST-OT not usable, by alpha (rows) and epsilon (columns):")
    print(d.assign(nc=~d.ok).pivot_table(index="alpha", columns="epsilon", values="nc", aggfunc="sum").to_string())
err = runs[runs.error.notna()][["pair", "run", "error"]]
print("\nerrors\n" + (err.to_string(index=False) if len(err) else "none"))

## 3. Growth distortion vs migration (the paper's Fig. 4B, per pair)

DeST-OT settings are coloured by α and shaped by γ; baselines in their own colours. The line marks each pair's Pareto front over all usable runs except the label reference.

In [ ]:
def front_mask(x, y):
    "Pareto-optimal points (both minimised)."
    order = np.lexsort((y, x))
    keep, best = np.zeros(len(x), bool), np.inf
    for i in order:
        if y[i] < best:
            keep[i], best = True, y[i]
    return keep


u = runs[runs.ok]
fig, axes = grid_axes(len(PAIRS))
for ax, p in zip(axes, PAIRS):
    r = u[u.pair == p]
    cmp_ = r[r.method != "label"]
    f = cmp_[front_mask(cmp_.migration.to_numpy(), cmp_.growth_dist_nt.to_numpy())].sort_values("migration")
    ax.plot(f.migration, f.growth_dist_nt, color=MUTED, lw=1, zorder=1)
    d = r[r.method == "destot"]
    for g, mk in GAMMA_MARK.items():
        dd = d[d.gamma == g]
        ax.scatter(dd.migration, dd.growth_dist_nt, c=dd.alpha, cmap=SEQ, vmin=0, vmax=1, marker=mk, s=22,
                   edgecolor="none", zorder=2)
    for m in ("expr", "fgw_bal", "moscot", "label"):
        b = r[r.method == m]
        col, mk, _ = METHOD[m]
        ax.scatter(b.migration, b.growth_dist_nt, color=col, marker=mk, s=40, edgecolor=SURFACE, lw=0.6, zorder=3,
                   alpha=0.5 if m == "label" else 1)
    ax.set_title(f"{p}  {pinfo.loc[p, 'a']} → {pinfo.loc[p, 'b']}", fontsize=8, color=INK)
    style(ax)
for ax in list(axes)[:1]:
    ax.set_ylabel("growth distortion (no transitions)", fontsize=8)
    ax.set_xlabel("migration (t1 RMS radii)", fontsize=8)
handles = [Line2D([], [], color=METHOD[m][0], marker=METHOD[m][1], ls="", label=METHOD[m][2]) for m in METHOD]
handles += [Line2D([], [], color=MUTED, marker=mk, ls="", label=f"γ = {g:g}") for g, mk in GAMMA_MARK.items()]
fig.legend(handles=handles, loc="lower center", ncol=6, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

## 4. Sensitivity of DeST-OT

Each metric, min–max normalised within a pair over the DeST-OT runs (0 = best of the pair, 1 = worst) and averaged over the pairs, at the main ε, over α × γ for each β.

In [ ]:
SCORE = {"growth_dist_nt": False, "migration": False, "kappa_ba": True, "gene_r": True}   # metric: higher is better


def normalised(df, cols=SCORE):
    "Per pair, min-max over the rows of df; 0 is best."
    out = df.copy()
    for c, higher in cols.items():
        g = df.groupby("pair")[c]
        lo, hi = g.transform("min"), g.transform("max")
        z = (df[c] - lo) / (hi - lo).replace(0, np.nan)
        out[c + "_n"] = (1 - z if higher else z).fillna(0)
    return out


d = normalised(runs[runs.ok & (runs.method == "destot")])
eps_main = d.epsilon.mode().iloc[0] if len(d) else np.nan
dm = d[d.epsilon == eps_main]
betas = sorted(dm.beta.unique())
fig, axes = plt.subplots(len(SCORE), len(betas), figsize=(3.2 * len(betas), 2.6 * len(SCORE)), squeeze=False)
for i, c in enumerate(SCORE):
    for j, b in enumerate(betas):
        ax = axes[i, j]
        t = dm[dm.beta == b].pivot_table(index="alpha", columns="gamma", values=c + "_n", aggfunc="mean")
        im = ax.imshow(t.to_numpy(), cmap=SEQ.reversed(), vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(t.shape[1]), [f"{g:g}" for g in t.columns], fontsize=7)
        ax.set_yticks(range(t.shape[0]), [f"{a:g}" for a in t.index], fontsize=7)
        for (y, x), v in np.ndenumerate(t.to_numpy()):
            ax.text(x, y, f"{v:.2f}", ha="center", va="center", fontsize=6, color=INK if v > 0.45 else SURFACE)
        ax.set_title(f"{c}, β = {b:g}", fontsize=8)
        ax.tick_params(length=0, colors=MUTED)
        if j == 0:
            ax.set_ylabel("α", fontsize=8)
        if i == len(SCORE) - 1:
            ax.set_xlabel("γ", fontsize=8)
fig.suptitle(f"normalised metric (0 = best of the pair; dark is good), ε = {eps_main:g}", fontsize=9)
fig.tight_layout()
plt.show()

## 5. Selection

The rule from the top: front membership (growth distortion × migration, tolerance `DELTA`) in at least `FRONT_MIN` of the pairs, then the best mean normalised κ_bal t2→t1 and gene r. Baselines are ranked by the same front share and label score among their own runs, for the comparison in `final`.

In [ ]:
def on_front(df, delta=DELTA):
    "Per pair: no other run better by more than delta in both normalised growth distortion and migration."
    out = pd.Series(False, index=df.index)
    for p, g in df.groupby("pair"):
        G, Mi = g.growth_dist_nt_n.to_numpy(), g.migration_n.to_numpy()
        dom = (G[None, :] < G[:, None] - delta) & (Mi[None, :] < Mi[:, None] - delta)
        out[g.index] = ~dom.any(1)
    return out


cand = normalised(runs[runs.ok & (runs.method != "label")])
cand["front"] = on_front(cand)
cand["labels_n"] = (cand.kappa_ba_n + cand.gene_r_n) / 2
n_pairs = cand.groupby("run").pair.nunique()
summary = cand.groupby(["method", "run"]).agg(pairs=("pair", "nunique"), front_share=("front", "mean"),
                                            labels_n=("labels_n", "mean"), growth_n=("growth_dist_nt_n", "mean"),
                                            migration_n=("migration_n", "mean"), kappa=("kappa_ba", "mean"),
                                            gene_r=("gene_r", "mean")).reset_index()
full = summary.pairs == len(PAIRS)
dest = summary[(summary.method == "destot") & full]
eligible = dest[dest.front_share >= FRONT_MIN]
if eligible.empty:
    eligible = dest[dest.front_share >= dest.front_share.quantile(0.75)]
    print(f"no setting is on the front of {FRONT_MIN:.0%} of the pairs; using the top quarter by front share")
pick = eligible.sort_values("labels_n", ascending=True).iloc[0] if len(eligible) else None
print("eligible DeST-OT settings (best label score first; *_n: 0 is best)")
print(eligible.sort_values("labels_n").head(12).round(3).to_string(index=False))
best_bl = (summary[(summary.method != "destot") & full].sort_values(["front_share", "labels_n"], ascending=[False, True])
           .groupby("method").head(1))
print("\nbest run of each baseline")
print(best_bl.round(3).to_string(index=False))
default = summary[summary.run == "destot_a0.2_b0.5_g50_e0.01"]
if len(default):
    print("\nthe paper's default (alpha 0.2, beta 0.5, gamma 50)")
    print(default.round(3).to_string(index=False))
if pick is not None:
    PICK = pick.run
    chosen = [PICK] + list(best_bl.run) + ([d for d in default.run if d != PICK])
    print(f"\nselected: {PICK}  ({tp.parse_run(PICK)[1]})")
    print("\nnext, on the GPU server:\n  python bench/scripts/temporal_align.py final --runs " + ",".join(chosen) + " --seeds 0,1")

In [ ]:
if pick is not None:
    sel = cand[cand.run.isin([PICK] + list(best_bl.run))]
    t = sel.pivot_table(index="pair", columns="run", values=["growth_dist_nt", "migration", "kappa_ba", "gene_r"])
    print("the selected setting and the best baselines, per pair")
    print(t.round(3).reindex(PAIRS).to_string())

## 6. The couplings of the selected setting

From `final/` (dense) when it exists, otherwise from the sweep's top-k columns. Per pair:
- **growth on t1:** log((1 + ρ) / area ratio): orange grows faster than the section as a whole, blue slower or shrinks;
- **ancestor organ on t2** (argmax of the t1 organ mass that flows into each t2 tile) next to the annotation;
- **organ transitions:** share of each t2 organ's ancestry from each t1 organ (columns sum to 1);
- **lines:** each t2 tile to its main ancestor (a subsample), t2 placed by the coupling's rigid fit.

In [ ]:
def coupling(pair, run, seed=SEED):
    "Dense coupling of a run (final if saved, else the sweep's top-k) and its source."
    f = OUT / "final" / pair / f"{run}_seed{seed}.npz"
    if f.exists():
        return np.load(f)["P"].astype(np.float64), "final"
    z = np.load(OUT / "couplings" / pair / f"{run}_seed{seed}.npz")
    return tp.dense_from_topk(z["idx"], z["vals"], int(z["n1"])), "top-k"


def masses_of(pair):
    a, b = pinfo.loc[pair, "a"], pinfo.loc[pair, "b"]
    na, nb = view[f"{a}|n_bins"].astype(np.float64), view[f"{b}|n_bins"].astype(np.float64)
    if settings.get("masses") == "uniform":
        return np.full(len(na), 1 / len(na)), np.full(len(nb), 1 / len(na))
    return na / na.sum(), nb / na.sum()


def organ_names(codes):
    return ORGANS[np.asarray(codes, int)]


def show_pair(pair, run):
    a, b = pinfo.loc[pair, "a"], pinfo.loc[pair, "b"]
    P, src = coupling(pair, run)
    xa, xb = view[f"{a}|xy"].astype(np.float64), view[f"{b}|xy"].astype(np.float64)
    oa, ob = organ_names(view[f"{a}|organ"]), organ_names(view[f"{b}|organ"])
    g1, g2 = masses_of(pair)
    rho = P.sum(1) / g1 - 1
    pal = organ_palette([oa, ob])
    # ancestor organ of each t2 tile: argmax of the organ mass of its column
    A = np.zeros((len(ORGANS), P.shape[1]))
    np.add.at(A, view[f"{a}|organ"].astype(int), P)
    anc = ORGANS[A.argmax(0)]
    R, t = tp.weighted_procrustes(xa, xb, P)
    xb_al = xb @ R + t

    fig, ax = plt.subplots(1, 5, figsize=(19, 3.8), gridspec_kw=dict(width_ratios=[1, 1, 1, 0.9, 1.2]))
    v = np.log1p(np.maximum(rho, -0.999)) - np.log(g2.sum() / g1.sum())     # relative to the whole section's growth
    lim = max(np.quantile(np.abs(v), 0.98), 1e-3)
    sc = ax[0].scatter(*xa.T, c=v, cmap=DIV, norm=TwoSlopeNorm(0, -lim, lim), s=3, lw=0)
    fig.colorbar(sc, ax=ax[0], shrink=0.7).ax.tick_params(labelsize=7)
    ax[0].set_title(f"{a} (t1): growth vs the area ratio ×{g2.sum() / g1.sum():.2f} (log)", fontsize=8)
    ax[1].scatter(*xb.T, c=[pal.get(o, OTHER) for o in ob], s=3, lw=0)
    ax[1].set_title(f"{b} (t2): annotation", fontsize=8)
    ax[2].scatter(*xb.T, c=[pal.get(o, OTHER) for o in anc], s=3, lw=0)
    ax[2].set_title(f"t2: ancestor organ ({(anc == ob).mean():.0%} same)", fontsize=8)
    for k in range(3):
        ax[k].set_aspect("equal")
        ax[k].axis("off")
    # organ transitions, organs with mass on both sides
    present = np.flatnonzero(metrics.SCORED & (np.bincount(view[f"{a}|organ"], minlength=len(ORGANS)) >= 5)
                             | metrics.SCORED & (np.bincount(view[f"{b}|organ"], minlength=len(ORGANS)) >= 5))
    Bm = np.zeros((len(ORGANS), len(ORGANS)))
    np.add.at(Bm, (view[f"{a}|organ"].astype(int)[:, None], view[f"{b}|organ"].astype(int)[None, :]), P)
    Tm = Bm[np.ix_(present, present)]
    Tm = Tm / np.where(Tm.sum(0) > 0, Tm.sum(0), 1)
    ax[3].imshow(Tm, cmap=SEQ, vmin=0, vmax=1)
    names = [ORGANS[i][:14] for i in present]
    ax[3].set_xticks(range(len(present)), names, rotation=90, fontsize=6)
    ax[3].set_yticks(range(len(present)), names, fontsize=6)
    ax[3].set_xlabel("t2 organ", fontsize=7)
    ax[3].set_ylabel("t1 ancestor organ", fontsize=7)
    ax[3].tick_params(length=0)
    ax[3].set_title("ancestry share per t2 organ", fontsize=8)
    # lines to the main ancestor
    rng = np.random.default_rng(0)
    js = rng.choice(P.shape[1], min(250, P.shape[1]), replace=False)
    i = P[:, js].argmax(0)
    off = np.array([1.2 * (np.ptp(xa[:, 0]) + np.ptp(xb_al[:, 0])) / 2, 0])
    ax[4].scatter(*xa.T, c=[pal.get(o, OTHER) for o in oa], s=2, lw=0)
    ax[4].scatter(*(xb_al + off).T, c=[pal.get(o, OTHER) for o in ob], s=2, lw=0)
    for jj, ii in zip(js, i):
        ax[4].plot([xa[ii, 0], xb_al[jj, 0] + off[0]], [xa[ii, 1], xb_al[jj, 1] + off[1]], color=INK, lw=0.25, alpha=0.35)
    ax[4].set_aspect("equal")
    ax[4].axis("off")
    ax[4].set_title(f"main ancestors (reflected fit: {np.linalg.det(R) < 0})", fontsize=8)
    fig.suptitle(f"{pair}: {run} [{src}]", fontsize=9, x=0.01, ha="left")
    fig.tight_layout()
    plt.show()


if pick is not None:
    for p in PAIRS:
        show_pair(p, PICK)

## 7. Growth rates against proliferation and apoptosis markers (the paper's Fig. 5)

Spearman ρ between the growth rate J = log(1 + ρ)/Δt of t1 tiles and their proliferation and apoptosis scores (moscot's marker sets, `scanpy.tl.score_genes`). A coupling that infers biological growth should correlate positively with proliferation and negatively with apoptosis. Balanced FGW has uniform growth and no correlation by construction.

In [ ]:
if pick is not None:
    show = [PICK] + [r for r in best_bl.run if not r.startswith("fgw_bal")]
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.4), sharey=True)
    for k, (col, name) in enumerate((("rho_J_prolif", "proliferation"), ("rho_J_apopt", "apoptosis"))):
        for n, r in enumerate(show):
            s = runs[(runs.run == r) & runs.ok].set_index("pair").reindex(PAIRS)
            m = tp.parse_run(r)[0]
            ax[k].scatter(np.arange(len(PAIRS)) + (n - len(show) / 2) * 0.12, s[col], color=METHOD[m][0], marker=METHOD[m][1],
                          s=28, label=r)
        ax[k].axhline(0, color=MUTED, lw=0.8)
        ax[k].set_xticks(range(len(PAIRS)), PAIRS, fontsize=8)
        ax[k].set_title(f"Spearman ρ(J, {name})", fontsize=9)
        style(ax[k])
    ax[0].legend(fontsize=7, frameon=False)
    fig.tight_layout()
    plt.show()

## 8. Seed stability of the final runs

Seed 1 re-tiles every section (Voronoi seed) and reruns the PCA. A setting whose metrics move a lot between seeds is fitting the tiling, not the tissue.

In [ ]:
if final is None or final.seed.nunique() < 2:
    print("no final runs with two seeds yet: run `temporal_align.py final ... --seeds 0,1`")
else:
    f = final[final.error.isna()]
    cols = ["growth_dist_nt", "migration", "kappa_ba", "gene_r"]
    fig, ax = plt.subplots(1, len(cols), figsize=(3.3 * len(cols), 3.2))
    for k, c in enumerate(cols):
        w = f.pivot_table(index=["pair", "run"], columns="seed", values=c)
        s0, s1 = sorted(f.seed.unique())[:2]
        for m in METHOD:
            ww = w[[tp.parse_run(r)[0] == m for r in w.index.get_level_values("run")]]
            ax[k].scatter(ww[s0], ww[s1], color=METHOD[m][0], marker=METHOD[m][1], s=22, label=METHOD[m][2])
        lo, hi = np.nanmin(w.to_numpy()), np.nanmax(w.to_numpy())
        ax[k].plot([lo, hi], [lo, hi], color=MUTED, lw=0.8)
        ax[k].set_title(c, fontsize=9)
        ax[k].set_xlabel(f"seed {s0}", fontsize=8)
        ax[k].set_ylabel(f"seed {s1}", fontsize=8)
        style(ax[k])
    ax[0].legend(fontsize=7, frameon=False)
    fig.tight_layout()
    plt.show()
    print(f.groupby("run")[cols].agg(["mean", "std"]).round(3).to_string())